# NAB Deep Anomaly Detection

This notebook inspects saved Phase 4 artifacts for the NAB deep anomaly-detection experiment. It does not train models. Reproduce the run with:

```powershell
.\.venv\Scripts\python.exe scripts\experiments\run_nab_deep_anomaly.py --run-id run_phase4_deep_anomaly
```

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

run_root = project_root / "experiments" / "anomaly" / "nab" / "deep_autoencoders"
run_dir = run_root / "run_phase4_deep_anomaly"
if not run_dir.exists():
    candidates = sorted(run_root.glob("run_*"))
    if not candidates:
        raise FileNotFoundError(
            "No NAB deep anomaly run found. Run scripts/experiments/run_nab_deep_anomaly.py first."
        )
    run_dir = candidates[-1]

run_dir

In [ ]:
def read_json(name: str) -> dict:
    with (run_dir / name).open("r", encoding="utf-8") as file:
        return json.load(file)


metrics = read_json("metrics.json")
dataset_summary = read_json("dataset_summary.json")
label_alignment = read_json("label_alignment.json")
ablation_results = read_json("ablation_results.json")
per_series = pd.read_csv(run_dir / "per_series_metrics.csv")
efficiency = pd.read_csv(run_dir / "efficiency.csv")

display(Markdown(f"## Run: `{run_dir.name}`"))
display(pd.DataFrame([dataset_summary]).T.rename(columns={0: "value"}))

## Label Alignment

The audit verifies that processed NAB labels can be mapped to observation timestamps before model comparison.

In [ ]:
alignment_fields = [
    "series_count",
    "observation_count",
    "labeled_series_count",
    "unlabeled_series_count",
    "anomaly_window_count",
    "point_label_count",
    "windows_without_observations",
    "windows_outside_observation_range",
    "point_labels_without_exact_match",
]
display(
    pd.DataFrame(
        {
            "field": alignment_fields,
            "value": [label_alignment[field] for field in alignment_fields],
        }
    )
)

## Global Metrics

All four models are evaluated on the same window-ending test timestamps.

In [ ]:
global_rows = []
for model_name, values in metrics["global"].items():
    delay = values["detection_delay"]
    global_rows.append(
        {
            "model": model_name,
            "precision": values["precision"],
            "recall": values["recall"],
            "f1": values["f1"],
            "pr_auc": values["pr_auc"],
            "false_positive_rate": values["false_positive_rate"],
            "detected_windows": delay["detected_windows"],
            "missed_windows": delay["missed_windows"],
            "median_delay_steps": delay["median_delay_steps"],
        }
    )

global_metrics = pd.DataFrame(global_rows).sort_values("f1", ascending=False)
display(global_metrics)

## Per-Series Robustness

In [ ]:
distribution_rows = []
for model_name, values in metrics["per_series_distribution"].items():
    positive_rows = per_series[(per_series["model"] == model_name) & (per_series["positives"] > 0)]
    distribution_rows.append(
        {
            "model": model_name,
            "mean_f1": values["f1"]["mean"],
            "median_f1": values["f1"]["median"],
            "f1_zero_series": values["f1_zero_series"],
            "positive_zero_f1_series": int((positive_rows["f1"] == 0).sum()),
            "mean_pr_auc": values["pr_auc"]["mean"],
        }
    )

display(pd.DataFrame(distribution_rows).sort_values("mean_f1", ascending=False))

In [ ]:
for model_name in sorted(per_series["model"].unique()):
    display(Markdown(f"### Best series by F1: `{model_name}`"))
    cols = ["entity_id", "precision", "recall", "f1", "pr_auc", "positives", "threshold"]
    display(
        per_series.loc[(per_series["model"] == model_name) & (per_series["positives"] > 0), cols]
        .sort_values(["f1", "recall", "precision"], ascending=False)
        .head(10)
    )

## Computational Cost

In [ ]:
efficiency_summary = pd.DataFrame(metrics["computational_efficiency"]).T.reset_index(names="model")
display(efficiency_summary.sort_values("training_seconds_total"))

## Ablation Study

The ablation uses a fixed six-series subset and trains Dense Autoencoder variants.

In [ ]:
ablation_frame = pd.DataFrame(ablation_results.get("results", []))
display(
    ablation_frame[
        [
            "variant",
            "sequence_length",
            "normalization",
            "feature_set",
            "threshold_strategy",
            "precision",
            "recall",
            "f1",
            "pr_auc",
            "false_positive_rate",
        ]
    ].sort_values("f1", ascending=False)
)

## LSTM Error Analysis

In [ ]:
error_analysis = read_json("error_analysis.json")
for section_name, rows in error_analysis["lstm_autoencoder"].items():
    display(Markdown(f"### {section_name}"))
    display(pd.DataFrame(rows).head(10))

## Figures

In [ ]:
for figure_path in sorted((run_dir / "figures").glob("*.png")):
    display(Markdown(f"### `{figure_path.name}`"))
    display(Image(filename=str(figure_path)))

## Notes

- Dense Autoencoder has the best global F1, precision, and recall in this run.
- Isolation Forest has the best global PR-AUC.
- LSTM Autoencoder improves over Isolation Forest on F1 and recall but costs substantially more than Dense AE.
- Median per-series F1 remains zero for all models, so robustness is still unresolved.